# Stage 4 — Circuit Construction
**Owner: Person A** | Week 3

## Goal
Build the sparse feature circuit: a directed graph where nodes are causally
important SAE features and edges are measured causal connections between layers.

## Prerequisite
Stage 3 must be complete. You need all_effects and important_features per layer.

In [ ]:
import sys; sys.path.append("..")
from src.config import get_config
cfg = get_config()

## 1. Load prerequisites from Stage 3

In [ ]:
# Load the outputs you saved at the end of 03_causal_features.ipynb
# all_effects, important_features_per_layer, feature_labels
# (save these as JSON at the end of notebook 03 so you can reload here)

## 2. Build the circuit

In [ ]:
from src.circuits import build_circuit
from src.model import get_model, preprocess_batch, get_device

device = get_device()
model = get_model()

# pixel_values from the flamingo/spoonbill dataset (same as notebook 03)

graph = build_circuit(
    important_features_per_layer=important_features_per_layer,
    model=model,
    pixel_values=pixel_values,
    feature_labels=feature_labels,
    ablation_effects=all_effects,
)

print(f"Circuit: {len(graph.nodes)} nodes, {len(graph.edges)} edges")

## 3. Save the circuit

In [ ]:
from src.circuits import save_circuit
path = save_circuit(graph)
print("Saved to:", path)

## 4. Faithfulness evaluation (Person C)

In [ ]:
from src.evaluate import evaluate_circuit_faithfulness

result = evaluate_circuit_faithfulness(
    graph=graph, model=model, pixel_values=pixel_values,
    class_a_idx=FLAMINGO_IDX, class_b_idx=SPOONBILL_IDX
)

print(result)
print(f"Faithfulness: {result['faithfulness']:.2%}")
print(f"Passes target (>= {cfg.circuit.faithfulness_target}): {result['passes_target']}")

## 5. Visualise the circuit (Person B)

In [ ]:
from src.visualise import plot_circuit_graph

fig = plot_circuit_graph(graph, save=False)
# When finalised: plot_circuit_graph(graph, save=True)

## 6. Comparison to baseline (Person C)

In [ ]:
from src.evaluate import compare_circuit_to_baseline

comparison = compare_circuit_to_baseline(
    graph=graph, model=model, pixel_values=pixel_values,
    class_a_idx=FLAMINGO_IDX, class_b_idx=SPOONBILL_IDX
)

print(comparison)
print(f"Circuit uses {comparison['circuit_nodes']} nodes vs. "
      f"{comparison['baseline_heads']} heads for same faithfulness")
print(f"Compression ratio: {comparison['compression_ratio']:.1f}x")